# 2026-10-19 Benchmark evidence checks
Run from within this repository. Sources and capture timestamps are in the linked JSON report. LiveBench compares model/suite grain after explicit field renaming; AA compares profile IDs; Epoch preserves every field except country-list ordering.


In [1]:
from pathlib import Path
import json, hashlib, csv, io, zipfile, collections
root = Path.cwd()
while not (root / 'docs/research/2026-10-19-benchmark-update-check.json').exists():
    if root == root.parent:
        raise FileNotFoundError('Run from within the benchmark repository')
    root = root.parent
report = json.loads((root / 'docs/research/2026-10-19-benchmark-update-check.json').read_text())
for item in report['evidenceFiles']:
    body = (root / item['path']).read_bytes()
    assert len(body) == item['byteLength']
    assert hashlib.sha256(body).hexdigest() == item['sha256']
print('Evidence files verified:', len(report['evidenceFiles']))


Evidence files verified: 261


In [2]:
base = root / 'artifacts/research/2026-10-18-benchmark-check'
current = root / 'artifacts/research/2026-10-19-benchmark-check'
def read_csv(path):
    return list(csv.DictReader(path.open()))
before = {r['model']: r for r in read_csv(base / 'live-epoch/table_2026_09_02.csv')}
after = {r['model']: r for r in read_csv(current / 'live-epoch/raw/table-current.csv')}
assert len(before) == len(after) == 48
assert after.keys() - before.keys() == {'deepseek-v4.1-flash-1017'}
assert before.keys() - after.keys() == {'smaug-flash'}
rename = {'long_horizon': 'long_agentic', 'design_layout': 'design'}
def normalized(row):
    return {rename.get(k,k): float(v) for k,v in row.items() if k != 'model'}
for model in sorted(before.keys() | after.keys()):
    old = normalized(before[model]) if model in before else None
    new = normalized(after[model]) if model in after else None
    if old != new:
        print(model, 'Overall', round(sum(old.values()) / len(old),1) if old else None,
              '->', round(sum(new.values()) / len(new),1) if new else None)
assert round(sum(normalized(after['deepseek-v4.1-flash-1017']).values()) / 7,1) == 79.2
assert (current / 'live-epoch/raw/table-current.csv').read_bytes() == (current / 'live-epoch/raw/table-current-bare.csv').read_bytes()


deepseek-v4.1-flash Overall 76.8 -> 78.0
deepseek-v4.1-flash-1017 Overall None -> 79.2
grok-4.6 Overall 74.6 -> 74.9
muse-spark-1.2 Overall 61.8 -> 62.4
muse-spark-1.3 Overall 74.9 -> 74.8
smaug-flash Overall 57.5 -> None


In [3]:
old_aa = {r['id']:r for r in json.loads((base / 'aa/models-rows.json').read_text())}
new_aa = {r['id']:r for r in json.loads((current / 'aa/models-rows.json').read_text())}
assert old_aa.keys() == new_aa.keys()
for key,row in new_aa.items():
    old = old_aa[key]
    assert old['intelligenceIndex'] == row['intelligenceIndex']
    if row['slug'] == 'deepseek-v4-1-flash':
        assert old['canonicalIntelligenceIndexTokenCount'] == row['canonicalIntelligenceIndexTokenCount']
        print('AA DeepSeek cost/task:',old['intelligenceIndexCostPerTask']['cost']['total'],'->',row['intelligenceIndexCostPerTask']['cost']['total'])
        print('AA performance source:',old['performanceDataSource'],'->',row['performanceDataSource'])
    for field in ['gdpval','briefcaseBreakdown']:
        if old.get(field) != row.get(field):
            print('AA score revision:',row['name'],field)
# Ignore only order within Country multi-value lists; all remaining ZIP fields must match.
def canonical_csv(blob):
    rows = list(csv.DictReader(io.StringIO(blob.decode('utf-8-sig'))))
    for row in rows:
        for key in ['Country','country']:
            if key in row:
                row[key] = ','.join(sorted(row[key].split(',')))
    return collections.Counter(json.dumps(r,sort_keys=True) for r in rows)
with zipfile.ZipFile(base / 'live-epoch/benchmark_data.zip') as old_zip, zipfile.ZipFile(current / 'live-epoch/raw/benchmark_data.zip') as new_zip:
    assert set(old_zip.namelist()) == set(new_zip.namelist())
    changed=[]
    for name in old_zip.namelist():
        a,b = old_zip.read(name),new_zip.read(name)
        if a != b:
            changed.append(name)
            assert name.endswith('.csv')
            assert canonical_csv(a) == canonical_csv(b),name
    print('Epoch ZIP changed files (Country order only):',changed)
print('Baseline:',report['baseline']['comparisonReport'])


AA DeepSeek cost/task: 0.26522527009606844 -> 0.485662249576018
AA performance source: {'type': 'firstParty', 'providerName': 'DeepSeek'} -> {'type': 'median'}
AA score revision: GLM-5.3-Flash briefcaseBreakdown
AA score revision: Kimi K3 (Max) gdpval
AA score revision: Kimi K3 (Max) briefcaseBreakdown
Epoch ZIP changed files (Country order only): ['gpqa_diamond.csv', 'video_mme_external.csv', 'bool_q_external.csv', 'gsm8k_external.csv', 'hella_swag_external.csv', 'model_metadata.csv']
Baseline: docs/research/2026-10-18-benchmark-update-check.json
